In [42]:

import pandas as pd 
from pathlib import Path
df = pd.read_csv("../FT Data - data.csv")
df.head()

,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [47]:
# here initially (rec_url_gcp) and (transcription_url_gcp) links are in this formate which is not working for us so we need to change it to the working formate that we can download the files using that link
# https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
# https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json

In [3]:
import requests
import io
import librosa
from datasets import Dataset, Audio
from tqdm import tqdm

In [4]:
# Function to ensure URLs are accessible based on your task instructions
def fix_url(url):
    # Example replacement logic if the 'here' link was broken
    return url.replace("old_link_prefix", "https://storage.googleapis.com/upload_goai")

df['transcription_url_gcp'] = df['transcription_url_gcp'].apply(fix_url)

In [ ]:
import requests

# CHECKING THE FIRST ROW OF THE DATAFRAME TO GET THE URLS FOR AUDIO AND TRANSCRIPTION
audio_url = "https://storage.googleapis.com/upload_goai/967179/825727_audio.wav"
json_url = "https://storage.googleapis.com/upload_goai/967179/825727_transcription.json"

def download_file(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download {url}. Status: {response.status_code}")

download_file(audio_url, "sample_audio1.wav")
download_file(json_url, "sample_transcript1.json")

Downloaded: sample_audio1.wav
Downloaded: sample_transcript1.json


In [50]:
# see we got audio and json file downloaded successfully  
# so now will extract all audio and json files using the same logic and then we will create a dataset in the formate that is required for whisper training

In [51]:
#From original df
df1 = df.loc[: ,"rec_url_gcp"]
df2 = df.loc[: ,"transcription_url_gcp"]

#Create new DataFrame
df3=pd.DataFrame({"audio_url":df1,"transcription_url":df2})

for i in range(len(df3)):  # Processing each row one by one
    temp1 = df3["audio_url"].iloc[i] #Get current URLs
    temp2 = df3["transcription_url"].iloc[i]
# now Split URL 
    sp1 = temp1.split("/")
    sp2 = temp2.split("/")

    base_url = "https://storage.googleapis.com/upload_goai/"

    df3["audio_url"].iloc[i] = base_url + sp1[-2] + "/" + sp1[-1]

    df3["transcription_url"].iloc[i] = base_url + sp2[-2] + "/" + sp2[-1]

df3.loc[:, "audio_url"]
df3.to_csv("fixed_urls.csv", index=False)

C:\Users\rishu\AppData\Local\Temp\ipykernel_16988\626723941.py:17: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df3["audio_url"].iloc[i] = base_url + sp1[-2] + "/" + sp1[-1]
C:\Users\rishu\AppData\Local\Temp\ipykernel_16988\626723941.py:19:

In [55]:
fixed_df = pd.read_csv("fixed_urls.csv")
fixed_df.head(3)

,audio_url,transcription_url
0,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
1,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
2,https://storage.googleapis.com/upload_goai/114...,https://storage.googleapis.com/upload_goai/114...


In [ ]:
# now we got same formate of URLs for all the audio and json files 
# now download all audio files and json files
# there are 104 total rows in original dataset 

In [30]:
from pathlib import Path
import requests

def download_file(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download {url}. Status: {response.status_code}")

audio_base_path = Path("../audio_data")
audio_base_path.mkdir(parents=True, exist_ok=True)

trans_base_path = Path("../transcript_data")
trans_base_path.mkdir(parents=True, exist_ok=True)

for i in range(len(df3)):
    audio_file = audio_base_path / f"sample_audio_{i}.wav"
    trans_file = trans_base_path / f"sample_transcript_{i}.json"

    download_file(df3.loc[i, "audio_url"], audio_file)
    download_file(df3.loc[i, "transcription_url"], trans_file)

Downloaded: ..\audio_data\sample_audio_0.wav
Downloaded: ..\transcript_data\sample_transcript_0.json
Downloaded: ..\audio_data\sample_audio_1.wav
Downloaded: ..\transcript_data\sample_transcript_1.json
Downloaded: ..\audio_data\sample_audio_2.wav
Downloaded: ..\transcript_data\sample_transcript_2.json
Downloaded: ..\audio_data\sample_audio_3.wav
Downloaded: ..\transcript_data\sample_transcript_3.json
Downloaded: ..\audio_data\sample_audio_4.wav
Downloaded: ..\transcript_data\sample_transcript_4.json
Downloaded: ..\audio_data\sample_audio_5.wav
Downloaded: ..\transcript_data\sample_transcript_5.json
Downloaded: ..\audio_data\sample_audio_6.wav
Downloaded: ..\transcript_data\sample_transcript_6.json
Downloaded: ..\audio_data\sample_audio_7.wav
Downloaded: ..\transcript_data\sample_transcript_7.json
Downloaded: ..\audio_data\sample_audio_8.wav
Downloaded: ..\transcript_data\sample_transcript_8.json
Downloaded: ..\audio_data\sample_audio_9.wav
Downloaded: ..\transcript_data\sample_transcri

In [ ]:
# all downloaded
# now we will create a dataset ready for Whisper training

In [41]:
import os
import json
import librosa
import soundfile as sf
import re
from pathlib import Path

AUDIO_DIR = Path("../audio_data")
TRANSCRIPT_DIR = Path("../transcript_data")
OUTPUT_DIR = Path("../dataset/chunks")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def clean_text(text):
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

dataset = []

for file in os.listdir(AUDIO_DIR):
    if not file.endswith(".wav"):
        continue
    
    # extract index
    index = file.replace("sample_audio_", "").replace(".wav", "")

    # build correct json filename
    json_file = f"sample_transcript_{index}.json"

    json_path = TRANSCRIPT_DIR / json_file

    print(f"Processing: {file}")

    if not os.path.exists(json_path):
        print(f"❌ Missing JSON for {file}")
        continue

    # Load audio
    y, sr = librosa.load(audio_path, sr=16000)

    # Load transcription
    with open(json_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    # Create folder for this audio
    audio_name = file.replace(".wav", "")
    audio_chunk_dir = os.path.join(OUTPUT_DIR, audio_name)
    os.makedirs(audio_chunk_dir, exist_ok=True)

    # Process segments
    for i, seg in enumerate(segments):
        start = int(seg["start"] * sr)
        end = int(seg["end"] * sr)

        chunk = y[start:end]

        if len(chunk) == 0:
            continue

        text = clean_text(seg["text"])
        if len(text) < 2:
            continue

        chunk_path = os.path.join(audio_chunk_dir, f"{i}.wav")
        sf.write(chunk_path, chunk, sr)

        dataset.append({
            "audio": chunk_path,
            "text": text
        })

# Save final dataset
with open("../dataset/dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print("✅ Dataset ready for Whisper training!")

Processing: sample_audio_0.wav
Processing: sample_audio_1.wav
Processing: sample_audio_10.wav
Processing: sample_audio_100.wav
Processing: sample_audio_101.wav
Processing: sample_audio_102.wav
Processing: sample_audio_103.wav
Processing: sample_audio_11.wav
Processing: sample_audio_12.wav
Processing: sample_audio_13.wav
Processing: sample_audio_14.wav
Processing: sample_audio_15.wav
Processing: sample_audio_16.wav
Processing: sample_audio_17.wav
Processing: sample_audio_18.wav
Processing: sample_audio_19.wav
Processing: sample_audio_2.wav
Processing: sample_audio_20.wav
Processing: sample_audio_21.wav
Processing: sample_audio_22.wav
Processing: sample_audio_23.wav
Processing: sample_audio_24.wav
Processing: sample_audio_25.wav
Processing: sample_audio_26.wav
Processing: sample_audio_27.wav
Processing: sample_audio_28.wav
Processing: sample_audio_29.wav
Processing: sample_audio_3.wav
Processing: sample_audio_30.wav
Processing: sample_audio_31.wav
Processing: sample_audio_32.wav
Processi

In [59]:
#now we have our dataset ready in the formate that is required for whisper training or fine-tuning

# {
#   "audio": "path/to/audio.wav",
#   "text": "transcribed sentence"
# }

examples: 

{

    "audio": "..\\dataset\\chunks\\sample_audio_0\\0.wav",

    "text": "अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब"
    
  },